# Demo phát hiện SQL Injection — Nhánh 1 + Nhánh 2

Load 2 model đã train (`models/branch1_v1/`, `models/branch2_v1/`), nhập một câu query/payload,
trả về kết quả kết hợp: nhãn Nhánh 1 (loại tấn công), điểm bất thường Nhánh 2, và verdict cuối cùng.

**Verdict logic (đơn giản hoá, chưa phải Bộ xử lý trung tâm đầy đủ):**
- Nhánh 1 = tấn công → **BLOCK**
- Nhánh 1 = normal + Nhánh 2 bất thường → **OVERKILL** (cần Admin xác nhận)
- Nhánh 1 = normal + Nhánh 2 bình thường → **ALLOW**


## 1. Load model

In [1]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
from src.preprocessing.canonicalize import canonicalize
from src.preprocessing.multiclass_tagger import LABEL_NAMES
from src.preprocessing.statistical_features import extract_statistical_features
from src.models.branch2_anomaly import AnomalyDetector
from src.utils import load_config

cfg = load_config('../../configs/config.yaml')

# Nhanh 1: TF-IDF + Logistic Regression
branch1_vectorizer = joblib.load('../../models/branch1_v1/vectorizer.joblib')
branch1_clf = joblib.load('../../models/branch1_v1/model.joblib')

# Nhanh 2: One-Class SVM (anomaly detector)
branch2_detector = AnomalyDetector.load('../../models/branch2_v1')

print('Da load xong Nhanh 1 (TF-IDF+LogReg) va Nhanh 2 (OCSVM)')


2026-07-27 23:03:45,413 | INFO     | src.models.branch2_anomaly | Model loaded from ../../models/branch2_v1


Da load xong Nhanh 1 (TF-IDF+LogReg) va Nhanh 2 (OCSVM)


## 2. Hàm `detect()` — nhập query, trả kết quả

In [2]:
def detect(query: str, anomaly_threshold: float = None) -> dict:
    """Chay 1 query qua Nhanh 1 + Nhanh 2, tra ve ket qua ket hop."""
    canonical = canonicalize(query, max_decode_iterations=cfg.get_path('preprocessing.max_decode_iterations', 3))
    text = canonical.query_canonical

    # --- Nhanh 1: supervised multiclass ---
    X1 = branch1_vectorizer.transform([text])
    label = int(branch1_clf.predict(X1)[0])
    proba = branch1_clf.predict_proba(X1)[0]
    label_name = LABEL_NAMES[label]
    branch1_confidence = float(proba[list(branch1_clf.classes_).index(label)])

    # --- Nhanh 2: anomaly detection (chi dua tren dac trung thong ke) ---
    feats = extract_statistical_features(text)
    X2 = np.array([feats.as_list()])
    anomaly_score = float(branch2_detector.score(X2)[0])
    is_anomaly = bool(branch2_detector.anomaly_flags(X2, threshold=anomaly_threshold)[0])

    # --- Verdict gop (logic don gian, chua phai Bo xu ly trung tam day du) ---
    if label != 0:  # 0 = normal
        verdict = 'BLOCK'
    elif is_anomaly:
        verdict = 'OVERKILL'
    else:
        verdict = 'ALLOW'

    return {
        'query_raw': query,
        'query_canonical': text,
        'has_comment_marker': canonical.has_comment_marker,
        'branch1_label': label_name,
        'branch1_confidence': round(branch1_confidence, 4),
        'branch2_anomaly_score': round(anomaly_score, 4),
        'branch2_is_anomaly': is_anomaly,
        'verdict': verdict,
    }


## 3. Thử với vài payload mẫu

In [3]:
test_queries = [
    ("SELECT * FROM users WHERE id = 42", 'benign - SELECT thuong'),
    ("1' OR '1'='1", 'SQLi - boolean-blind kinh dien'),
    ("1' UNION SELECT username, password FROM users--", 'SQLi - union-based'),
    ("' AND SLEEP(5)--", 'SQLi - time-blind'),
    ("admin'--", 'SQLi - comment injection'),
    ("john.doe@example.com", 'benign - email nhu binh thuong'),
]

for query, note in test_queries:
    result = detect(query)
    print(f"[{note}]")
    print(f"  input: {query!r}")
    print(f"  Nhanh 1: {result['branch1_label']} (conf={result['branch1_confidence']})")
    print(f"  Nhanh 2: anomaly_score={result['branch2_anomaly_score']} is_anomaly={result['branch2_is_anomaly']}")
    print(f"  => VERDICT: {result['verdict']}")
    print()


[benign - SELECT thuong]
  input: 'SELECT * FROM users WHERE id = 42'
  Nhanh 1: normal (conf=0.954)
  Nhanh 2: anomaly_score=-4.6433 is_anomaly=False
  => VERDICT: ALLOW

[SQLi - boolean-blind kinh dien]
  input: "1' OR '1'='1"
  Nhanh 1: boolean_blind (conf=0.9995)
  Nhanh 2: anomaly_score=-4.6437 is_anomaly=False
  => VERDICT: BLOCK

[SQLi - union-based]
  input: "1' UNION SELECT username, password FROM users--"
  Nhanh 1: union_based (conf=0.6974)
  Nhanh 2: anomaly_score=-4.3742 is_anomaly=False
  => VERDICT: BLOCK

[SQLi - time-blind]
  input: "' AND SLEEP(5)--"
  Nhanh 1: time_blind (conf=1.0)
  Nhanh 2: anomaly_score=-4.6597 is_anomaly=False
  => VERDICT: BLOCK

[SQLi - comment injection]
  input: "admin'--"
  Nhanh 1: boolean_blind (conf=0.55)
  Nhanh 2: anomaly_score=-3.1409 is_anomaly=False
  => VERDICT: BLOCK

[benign - email nhu binh thuong]
  input: 'john.doe@example.com'
  Nhanh 1: normal (conf=0.9233)
  Nhanh 2: anomaly_score=-2.8913 is_anomaly=False
  => VERDICT: ALLOW

## 4. Nhập tay (thử query của riêng bạn)

Đổi giá trị `my_query` bên dưới rồi chạy cell để test.

In [4]:
my_query = "' OR 1=1 -- "  # <-- doi cho nay
detect(my_query)


{'query_raw': "' OR 1=1 -- ",
 'query_canonical': "' or 1=1 -- ",
 'has_comment_marker': 1,
 'branch1_label': 'boolean_blind',
 'branch1_confidence': 0.9997,
 'branch2_anomaly_score': -4.6338,
 'branch2_is_anomaly': False,
 'verdict': 'BLOCK'}

## 5. (Tuỳ chọn) Đánh giá nhanh trên tập test Nhánh 1

Chạy `detect()` trên một phần test set thật, so với nhãn gốc, để có con số accuracy nhanh
(không thay thế báo cáo `reports/branch1_eval.json` đầy đủ, chỉ để sanity-check trực quan).

In [5]:
import pandas as pd

df_test = pd.read_csv('../../data/processed/branch1_train.csv')
df_test = df_test[df_test['split'] == 'test'].sample(20, random_state=42)

correct = 0
for _, row in df_test.iterrows():
    pred = detect(row['query_raw'])
    ok = pred['branch1_label'] == row['label_name']
    correct += ok
    if not ok:
        print(f"SAI: true={row['label_name']} pred={pred['branch1_label']} | {row['query_raw'][:60]!r}")

print(f"\nDung {correct}/{len(df_test)} tren mau 20 dong ngau nhien tu test set")


SAI: true=normal pred=boolean_blind | '/blog";sleep 15;"/wp-login.php?reauth=1&redirect_to=http://t'

Dung 19/20 tren mau 20 dong ngau nhien tu test set


<!-- branch1-arch-compare -->
## 6. So sánh 3 kiến trúc Nhánh 1 (TF-IDF+LogReg vs CNN vs DistilBERT)

Ngoài model production (`branch1_v1` = TF-IDF + LogReg), phần này nạp thêm **2 model candidate**
đã huấn luyện trên **cùng dữ liệu 5 lớp** bằng `train/compare_branch1_architectures.py`:
CNN + SQL-tokenizer và DistilBERT fine-tune. Weights lấy từ Hugging Face
(`Jason-42195/VNU-SQLi-Detection-Models`, thư mục `branch1_comparison/`) hoặc chạy lại script:

```bash
hf download Jason-42195/VNU-SQLi-Detection-Models branch1_comparison/ \
    --local-dir models/ --repo-type model
```


In [6]:
import json as _json
from pathlib import Path
import torch

sys.path.insert(0, '..')  # train/ (chua compare_branch1_architectures.py)  # de import helper tu script so sanh kien truc
from compare_branch1_architectures import _build_textcnn_class, _encode_texts

_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_COMP_DIR = Path('../../models/branch1_comparison')
_cnn_cfg = cfg.get_path('branch1_supervised.cnn_sqltok')

# --- CNN + SQL-tokenizer ---
_cnn_dir = _COMP_DIR / 'candidate_cnn_sqltok'
with (_cnn_dir / 'vocab.json').open(encoding='utf-8') as f:
    cnn_vocab = _json.load(f)
_cnn_state = torch.load(_cnn_dir / 'model.pt', map_location=_device)
_cnn_nclass = int(_cnn_state['fc.bias'].shape[0])  # so lop lay tu chinh model head
TextCNN = _build_textcnn_class()
cnn_model = TextCNN(
    vocab_size=len(cnn_vocab), embedding_dim=_cnn_cfg['embedding_dim'],
    num_filters=_cnn_cfg['num_filters'], kernel_sizes=_cnn_cfg['kernel_sizes'],
    num_classes=_cnn_nclass,
).to(_device)
cnn_model.load_state_dict(_cnn_state)
cnn_model.eval()

# --- DistilBERT fine-tune ---
from transformers import AutoModelForSequenceClassification, AutoTokenizer
_bert_dir = _COMP_DIR / 'candidate_distilbert'
bert_tok = AutoTokenizer.from_pretrained(str(_bert_dir))
bert_model = AutoModelForSequenceClassification.from_pretrained(str(_bert_dir)).to(_device)
bert_model.eval()
_bert_maxlen = cfg.get_path('branch1_supervised.distilbert.max_length', 128)

print(f'Loaded CNN ({_cnn_nclass}-class, {sum(p.numel() for p in cnn_model.parameters()):,} params) '
      f'+ DistilBERT ({bert_model.config.num_labels}-class) on {_device}')


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded CNN (5-class, 28,485 params) + DistilBERT (5-class) on cuda


In [7]:
def branch1_variants(query: str) -> dict:
    """Chay 1 query qua ca 3 kien truc Nhanh 1, tra ve (nhan, do_tin_cay) cho tung model."""
    text = canonicalize(
        query, max_decode_iterations=cfg.get_path('preprocessing.max_decode_iterations', 3)
    ).query_canonical

    # TF-IDF + LogReg (production branch1_v1)
    p1 = branch1_clf.predict_proba(branch1_vectorizer.transform([text]))[0]
    i1 = int(p1.argmax())
    tfidf = (LABEL_NAMES[int(branch1_clf.classes_[i1])], round(float(p1[i1]), 4))

    # CNN + SQL-tokenizer
    with torch.no_grad():
        x = _encode_texts([text], cnn_vocab, _cnn_cfg['max_length']).to(_device)
        pc = torch.softmax(cnn_model(x), dim=-1)[0].cpu()
    ic = int(pc.argmax()); cnn = (LABEL_NAMES[ic], round(float(pc[ic]), 4))

    # DistilBERT
    with torch.no_grad():
        enc = bert_tok([text], padding=True, truncation=True,
                       max_length=_bert_maxlen, return_tensors='pt').to(_device)
        pb = torch.softmax(bert_model(**enc).logits, dim=-1)[0].cpu()
    ib = int(pb.argmax()); bert = (LABEL_NAMES[ib], round(float(pb[ib]), 4))

    return {'query': query, 'tfidf_logreg': tfidf, 'cnn_sqltok': cnn, 'distilbert': bert}


In [8]:
import pandas as pd

_rows = []
for _query, _note in test_queries:
    _r = branch1_variants(_query)
    _rows.append({
        'note': _note,
        'query': _query,
        'tfidf_logreg': f'{_r["tfidf_logreg"][0]} ({_r["tfidf_logreg"][1]})',
        'cnn_sqltok':   f'{_r["cnn_sqltok"][0]} ({_r["cnn_sqltok"][1]})',
        'distilbert':   f'{_r["distilbert"][0]} ({_r["distilbert"][1]})',
    })
pd.DataFrame(_rows)


,note,query,tfidf_logreg,cnn_sqltok,distilbert
0,benign - SELECT thuong,SELECT * FROM users WHERE id = 42,normal (0.954),normal (0.9997),normal (0.9993)
1,SQLi - boolean-blind kinh dien,1' OR '1'='1,boolean_blind (0.9995),boolean_blind (0.9981),boolean_blind (0.9995)
2,SQLi - union-based,"1' UNION SELECT username, password FROM users--",union_based (0.6974),union_based (0.862),union_based (0.9999)
3,SQLi - time-blind,' AND SLEEP(5)--,time_blind (1.0),time_blind (1.0),boolean_blind (0.5073)
4,SQLi - comment injection,admin'--,boolean_blind (0.55),boolean_blind (0.9697),boolean_blind (0.9982)
5,benign - email nhu binh thuong,john.doe@example.com,normal (0.9233),normal (0.9934),normal (0.9996)


### Thu query cua rieng ban qua ca 3 model

Doi `my_query` roi chay cell.

In [9]:
branch1_variants("' OR 1=1 -- ")

{'query': "' OR 1=1 -- ",
 'tfidf_logreg': ('boolean_blind', 0.9997),
 'cnn_sqltok': ('boolean_blind', 0.9971),
 'distilbert': ('boolean_blind', 0.9997)}